In [ ]:
from datetime import date
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

df = catalog.load('raw/openaire/researchproduct_dev#parquet')

In [ ]:
df

In [ ]:
def _pick_load_dt(df: pd.DataFrame):
    # Si hay una sola fecha en el batch, usala; si hay varias, quedate con la más reciente;
    # si no hay, hoy.
    if 'load_datetime' not in df.columns or df['_load_datetime'].isna().all():
        return date.today()
    vals = df['_load_datetime'].dropna()
    if vals.nunique() == 1:
        return vals.iloc[0]
    return pd.to_datetime(vals).max().date()

# Transformaciones

In [ ]:
df_research_collectedfrom = df[['id','collectedFrom']].explode('collectedFrom').reset_index(drop=True)
df_research_collectedfrom.rename(columns={'id':'researchproduct_id'}, inplace=True)

In [ ]:
df_research_collectedfrom

In [ ]:
df_research_collectedfrom['collectedFrom'] \
    .dropna() \
    .map(lambda d: sorted(d.keys())) \
    .explode() \
    .value_counts()

In [ ]:
df_collectedfrom = pd.json_normalize(df_research_collectedfrom['collectedFrom'])
df_collectedfrom.rename(columns={'key':'datasource_id'}, inplace=True)

In [ ]:
df_collectedfrom

## Paso 1: Convierto tipos y selecciono columnas con cardinalidad 1 con respecto a cada research product
+ info en https://graph.openaire.eu/docs/data-model/entities/research-product

In [ ]:
def openaire_land_researchproduct_collectedfrom(df: pd.DataFrame)-> pd.DataFrame:

    load_dt = _pick_load_dt(df)

    df_research_collectedfrom = df[['id','collectedFrom']].explode('collectedFrom').reset_index(drop=True)
    df_research_collectedfrom.rename(columns={'id':'researchproduct_id'}, inplace=True)

    df_collectedfrom = pd.json_normalize(df_research_collectedfrom['collectedFrom'])
    df_collectedfrom.rename(columns={'key':'datasource_id'}, inplace=True)

    df_research_collectedfrom = pd.concat(
        [df_research_collectedfrom['researchproduct_id'], df_collectedfrom.loc[:,['datasource_id','value']]], 
        axis=1
    )

    df_research_collectedfrom['_load_datetime'] = date.today()

    return df_research_collectedfrom


In [ ]:
df_research_collectedfrom = openaire_land_researchproduct_collectedfrom(df)

In [ ]:
df_research_collectedfrom